In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()

def find_release_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        data_root = candidate / 'Step0_Data' / '1_Data'
        if (data_root / 'WDI' / 'WDIEXCEL-2026_02_01.xlsx').is_file():
            return candidate
    raise FileNotFoundError('Could not locate standalone release root from the current working directory')

release_root = find_release_root(cwd)
shared_data_root = release_root / 'Step0_Data' / '1_Data'
step_root = release_root / 'Step1_FeaturePool'
step0_output = release_root / 'Step0_Data' / '3_Results' / '0_cleaned_baseline.csv'
wdi_dir = shared_data_root / 'WDI'
scan_results_path = step_root / '2_Results' / '0_wdi_all_indicators_coverage.csv'
wdi_path = wdi_dir / 'WDIEXCEL-2026_02_01.xlsx'
scope_path = wdi_dir / 'WDI_182countries_1990_2024_2017PPP.xlsx'
output_path = step_root / '2_Results' / '1_selected_features_data.csv'
selected_codes = [
    'SP.POP.TOTL', 'SP.POP.GROW', 'SP.URB.TOTL.IN.ZS', 'SP.URB.GROW',
    'SP.DYN.LE00.IN', 'EG.ELC.ACCS.ZS', 'EN.ATM.PM25.MC.M3', 'NV.IND.TOTL.ZS',
    'NV.SRV.TOTL.ZS', 'EN.POP.DNST', 'NY.GDP.MKTP.KD.ZG', 'EN.GHG.CO2.MT.CE.AR5',
]
years = [str(y) for y in range(1990, 2025)]
possible_iso_cols = ['ISO3', 'ISO3 Code', 'Country Code', 'Code']

In [ ]:
scan_df = pd.read_csv(scan_results_path)
missing_in_scan = [code for code in selected_codes if code not in set(scan_df['Indicator Code'].astype(str))]
missing_in_scan

In [ ]:
scope_df = pd.read_excel(scope_path, sheet_name='GDP_PPP_2017')
iso_col = next((col for col in possible_iso_cols if col in scope_df.columns), scope_df.columns[0])
target_iso_list = scope_df[iso_col].dropna().astype(str).unique().tolist()
wdi_full = pd.read_excel(wdi_path, sheet_name='Data')
column_lookup = {str(col): col for col in wdi_full.columns}
year_cols = [column_lookup[year] for year in years]
wdi_target = wdi_full[
    wdi_full['Country Code'].astype(str).isin(target_iso_list)
    & wdi_full['Indicator Code'].astype(str).isin(selected_codes)
].copy()
wdi_target.shape

In [ ]:
wdi_long = pd.melt(
    wdi_target,
    id_vars=['Country Code', 'Indicator Code'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Value',
)
wdi_long['Year'] = wdi_long['Year'].astype(int)
wdi_panel = wdi_long.pivot(index=['Country Code', 'Year'], columns='Indicator Code', values='Value').reset_index()
gdp_wide = scope_df.rename(columns={iso_col: 'Country Code'}).copy()
gdp_long = gdp_wide.melt(id_vars=['Country Code'], value_vars=years, var_name='Year', value_name='NY.GDP.MKTP.PP.KD')
gdp_long['Year'] = gdp_long['Year'].astype(int)
wdi_panel = wdi_panel.merge(gdp_long, on=['Country Code', 'Year'], how='left')
wdi_panel['NY.GDP.PCAP.PP.KD'] = wdi_panel['NY.GDP.MKTP.PP.KD'] / wdi_panel['SP.POP.TOTL']
wdi_panel.shape

In [ ]:
baseline = pd.read_csv(step0_output).rename(columns={'ISO3': 'Country Code'})
baseline = baseline[['Country Code', 'Year', 'Country', 'IncomeGroup', 'AW_t', 'CDW_t', 'IW_t', 'MSW_t']].copy()
region_lookup = pd.read_excel(wdi_path, sheet_name='Country')[['Country Code', 'Region']].drop_duplicates()
selected_panel = wdi_panel.merge(baseline, on=['Country Code', 'Year'], how='left')
selected_panel = selected_panel.merge(region_lookup, on='Country Code', how='left')
selected_panel = selected_panel.rename(columns={'Country': 'Country Name'})
output_path.parent.mkdir(parents=True, exist_ok=True)
selected_panel.to_csv(output_path, index=False)
selected_panel.head()